# Household Electricity Bill Increase Classification — MLP

This notebook contains the MLP classification work contributed to the three-member research project.

**Personal contribution:** MLP modeling

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

## 1. Load Processed Data

In [2]:
df = pd.read_csv("../data/processed_household_survey.csv")

X = df.drop(columns=["label_kenaikan"])
y = df["label_kenaikan"].astype(int)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(y.value_counts())

Dataset shape: (286, 49)

Class distribution:
label_kenaikan
0    223
1     63
Name: count, dtype: int64


## 2. Train-Test Split and SMOTE

SMOTE is applied **only to the training data**, while the test set remains untouched for evaluation.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Original training distribution:")
print(y_train.value_counts())

print("\nTraining distribution after SMOTE:")
print(y_train_res.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 3. MLP Experiment

In [ ]:
from sklearn.neural_network import MLPClassifier

configs = [
    (16,),
    (32,),
    (32, 8),
    (32, 16),
    (32, 32),
    (64, 32),
    (64, 32, 16)
]

results = []

for cfg in configs:
    mlp = MLPClassifier(
        hidden_layer_sizes=cfg,
        activation='relu',
        solver='adam',
        alpha=0.0005,
        learning_rate_init=0.001,
        max_iter=800,
        random_state=42
    )

    mlp.fit(X_train_res, y_train_res)
    y_pred_cfg = mlp.predict(X_test)

    results.append({
        "configuration": str(cfg),
        "accuracy": accuracy_score(y_test, y_pred_cfg),
        "precision": precision_score(y_test, y_pred_cfg, zero_division=0),
        "recall": recall_score(y_test, y_pred_cfg, zero_division=0),
        "f1": f1_score(y_test, y_pred_cfg, zero_division=0)
    })

results_df = pd.DataFrame(results)
display(results_df)

best_idx = results_df["f1"].idxmax()
best_cfg = configs[best_idx]

print("Best configuration by F1:", best_cfg)

mlp = MLPClassifier(
    hidden_layer_sizes=best_cfg,
    activation='relu',
    solver='adam',
    alpha=0.0005,
    learning_rate_init=0.001,
    max_iter=800,
    random_state=42
)

mlp.fit(X_train_res, y_train_res)
y_pred = mlp.predict(X_test)

## 4. Evaluation Summary

The classification report is the primary evaluation output used in the original experiment.

In [ ]:
print(classification_report(y_test, y_pred))

print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score :", round(f1_score(y_test, y_pred, zero_division=0), 4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["No Increase", "Increase"],
    yticklabels=["No Increase", "Increase"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"{model_name} Confusion Matrix")
plt.show()